# Extraction Analysis

Recovery (recall/precision vs. ground truth) and hallucination rate for a single extraction run.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT / 'experiments'))
sys.path.insert(0, str(REPO_ROOT))

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
DATASET         = 'pond'
MODEL           = 'gemma-3-27b'
EXTRACTION_DATE = '2026_04_01'  # set to None to use most recent run

In [ ]:
import pandas as pd
from analysis.loaders import load_extraction, load_combined_judgements, load_ground_truth
from analysis.metrics import recovery_rate, hallucination_rate, per_paper_metrics
from analysis.plots import recovery_bar, probability_distribution
from experiments.run_extraction import load_dataset_config

config = load_dataset_config(DATASET)

## Load data

In [ ]:
records = load_extraction(DATASET, MODEL, EXTRACTION_DATE)
extraction_df = pd.DataFrame(records)
print(f'{len(extraction_df)} extracted measurements')
extraction_df.head()

In [ ]:
ground_truth_df = load_ground_truth(config)
print(f'{len(ground_truth_df)} ground truth measurements')
ground_truth_df.head()

In [ ]:
judgements = load_combined_judgements(DATASET, MODEL, EXTRACTION_DATE)
judged_df = pd.DataFrame(judgements)
print(f'{len(judged_df)} judged measurements')

## Recovery

In [ ]:
# Update strict_matching to match your dataset's entity key columns
STRICT_MATCHING = {'entity': ['name', 'location']}
FUZZY_MATCHING  = {'value': ['value']}   # or {} for strict-only

stats = recovery_rate(
    extraction_df,
    ground_truth_df,
    strict_matching=STRICT_MATCHING,
    fuzzy_matching=FUZZY_MATCHING,
)
print(stats)

## Hallucination rate

In [ ]:
hall = hallucination_rate(judged_df)
print(hall)

fig = probability_distribution(judged_df, prob_col='judgement_p_true', label_col='judgement_combined')
fig.savefig('figures/prob_distribution.pdf', bbox_inches='tight')
fig.show()

## Per-paper breakdown

In [ ]:
paper_df = per_paper_metrics(
    extraction_df,
    ground_truth_df,
    judged_df=judged_df,
    strict_matching=STRICT_MATCHING,
    fuzzy_matching=FUZZY_MATCHING,
)
paper_df.sort_values('recall').round(3)